# Week 6 &mdash; Final Project

## Implementation: Mesh Part-Segmentation with a Graph Neural Network

This is the complete, runnable capstone pipeline. We (1) generate a synthetic labelled mesh dataset, (2) compute **intrinsic geometric features** (Heat Kernel Signature + Laplace&ndash;Beltrami eigenfunctions, from Week 5), (3) build a mesh graph, (4) train a **GNN segmenter** (Week 3) in PyTorch Geometric, and (5) evaluate with accuracy and **mIoU** and visualise the results.

### Setup


In [ ]:
# !pip install numpy scipy torch torch_geometric matplotlib

import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 1. A Synthetic Labelled Mesh Dataset

We generate **cylinder-with-caps** meshes and label each vertex by semantic part &mdash; *bottom cap*, *side wall*, *top cap* &mdash; giving a clean 3-class segmentation problem. Each mesh is randomly deformed (height/radius), so the model must learn *shape*-based rules, not memorise coordinates.


In [ ]:
def make_labelled_mesh(nu=24, nv=14, radius=1.0, height=2.0):
    """Cylinder side + two caps. Returns (V, F, labels in {0,1,2})."""
    us = np.linspace(0, 2*np.pi, nu, endpoint=False)
    zs = np.linspace(0, height, nv)
    verts, labels = [], []
    grid = np.zeros((nu, nv), dtype=int)
    for i, u in enumerate(us):
        for j, z in enumerate(zs):
            grid[i, j] = len(verts)
            verts.append([radius*np.cos(u), radius*np.sin(u), z])
            # label by vertical position: bottom / side / top
            if j == 0:        labels.append(0)
            elif j == nv-1:   labels.append(2)
            else:             labels.append(1)
    verts = np.array(verts)
    faces = []
    for i in range(nu):
        for j in range(nv-1):
            a, b = grid[i, j], grid[(i+1) % nu, j]
            c, d = grid[(i+1) % nu, j+1], grid[i, j+1]
            faces.append([a, b, c]); faces.append([a, c, d])
    return verts, np.array(faces), np.array(labels)

V, Fc, y = make_labelled_mesh()
print(f"Vertices: {len(V)} | Faces: {len(Fc)} | Classes: {sorted(set(y))}")


In [ ]:
def plot_mesh_labels(V, Fc, labels, title, ax):
    cmap = np.array([[0.90,0.30,0.24],[0.18,0.55,0.34],[0.20,0.40,0.85]])
    face_lab = np.array([np.bincount(labels[f], minlength=3).argmax() for f in Fc])
    polys = [V[f] for f in Fc]
    coll = Poly3DCollection(polys, facecolors=cmap[face_lab], edgecolors="k", linewidths=0.1)
    ax.add_collection3d(coll)
    for d in range(3):
        ax.set_xlim(V[:,0].min(),V[:,0].max())
        ax.set_ylim(V[:,1].min(),V[:,1].max())
        ax.set_zlim(V[:,2].min(),V[:,2].max())
    ax.set_box_aspect((np.ptp(V[:,0]),np.ptp(V[:,1]),np.ptp(V[:,2])))
    ax.set_title(title); ax.axis("off")

fig = plt.figure(figsize=(4,4))
ax = fig.add_subplot(111, projection="3d")
plot_mesh_labels(V, Fc, y, "Ground-truth parts\n(red=bottom, green=side, blue=top)", ax)
plt.tight_layout(); plt.show()


## 2. Intrinsic Features &mdash; the Cotangent Laplacian (Week 5)

We reuse the cotangent Laplace&ndash;Beltrami operator to derive **isometry-invariant** node features.


In [ ]:
def cotangent_laplacian(V, F):
    n = len(V); C = sp.lil_matrix((n, n)); mass = np.zeros(n)
    for tri in F:
        i, j, k = tri
        vi, vj, vk = V[i], V[j], V[k]
        def cot(a, b):
            cr = np.linalg.norm(np.cross(a, b))
            return np.dot(a, b)/cr if cr > 1e-12 else 0.0
        cot_k = cot(vi-vk, vj-vk); cot_i = cot(vj-vi, vk-vi); cot_j = cot(vi-vj, vk-vj)
        for a, b, w in [(i,j,cot_k),(j,k,cot_i),(i,k,cot_j)]:
            C[a,b]-=0.5*w; C[b,a]-=0.5*w; C[a,a]+=0.5*w; C[b,b]+=0.5*w
        area = 0.5*np.linalg.norm(np.cross(vj-vi, vk-vi))
        mass[[i,j,k]] += area/3.0
    return C.tocsr(), sp.diags(mass)

def intrinsic_features(V, F, k_eig=20, n_hks=10):
    C, M = cotangent_laplacian(V, F)
    k = min(k_eig, len(V)-2)
    ev, ef = spla.eigsh(C, k=k, M=M, sigma=0, which="LM")
    o = np.argsort(ev); ev, ef = ev[o], ef[:, o]
    # Heat Kernel Signature across log-spaced time scales.
    ts = np.logspace(-2, 1, n_hks)
    hks = np.stack([(np.exp(-ev[None,:]*t) * ef**2).sum(1) for t in ts], axis=1)
    # Normalise each feature column for stable training.
    feats = np.concatenate([hks, ef[:, 1:6]], axis=1)   # HKS + low eigenfunctions
    feats = (feats - feats.mean(0)) / (feats.std(0) + 1e-8)
    return feats.astype(np.float32)

X = intrinsic_features(V, Fc)
print("Per-vertex feature matrix:", X.shape, "(HKS scales + eigenfunction encodings)")


## 3. Building the Mesh Graph (PyTorch Geometric)

We convert mesh edges into an `edge_index` and assemble a `Data` object with features, labels, and train/val/test masks over vertices.


In [ ]:
from torch_geometric.data import Data

def mesh_to_graph(V, F, X, y):
    edges = set()
    for a, b, c in F:
        for u, v in [(a,b),(b,c),(a,c)]:
            edges.add((u, v)); edges.add((v, u))
    edge_index = torch.tensor(list(edges), dtype=torch.long).t().contiguous()
    data = Data(x=torch.tensor(X), edge_index=edge_index,
                y=torch.tensor(y, dtype=torch.long))
    n = len(V); idx = np.random.permutation(n)
    tr, va = int(0.6*n), int(0.8*n)
    masks = {"train": idx[:tr], "val": idx[tr:va], "test": idx[va:]}
    for name, ids in masks.items():
        m = torch.zeros(n, dtype=torch.bool); m[ids] = True
        setattr(data, f"{name}_mask", m)
    return data

data = mesh_to_graph(V, Fc, X, y).to(device)
print(data)


## 4. The GNN Segmenter (Week 3)

A three-layer GCN maps intrinsic node features to per-vertex class logits. Because each layer is permutation-equivariant, the whole network is &mdash; exactly what per-vertex segmentation requires.


In [ ]:
from torch_geometric.nn import GCNConv

class MeshSegGNN(nn.Module):
    def __init__(self, in_dim, hidden, n_classes):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.conv3 = GCNConv(hidden, n_classes)
        self.drop = nn.Dropout(0.3)
    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index)); x = self.drop(x)
        x = F.relu(self.conv2(x, edge_index)); x = self.drop(x)
        return self.conv3(x, edge_index)

model = MeshSegGNN(data.num_features, 64, 3).to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
print(model)


## 5. Training


In [ ]:
def train_step():
    model.train(); opt.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward(); opt.step(); return float(loss)

@torch.no_grad()
def accuracy(mask):
    model.eval()
    pred = model(data.x, data.edge_index).argmax(1)
    return float((pred[mask] == data.y[mask]).float().mean())

for epoch in range(1, 201):
    loss = train_step()
    if epoch % 40 == 0:
        print(f"Epoch {epoch:3d} | loss {loss:.4f} | "
              f"train {accuracy(data.train_mask):.3f} | "
              f"val {accuracy(data.val_mask):.3f} | "
              f"test {accuracy(data.test_mask):.3f}")


## 6. Evaluation &mdash; Accuracy and mIoU

We implement the **mean Intersection-over-Union** metric from the project brief and report it alongside accuracy on the test vertices.


In [ ]:
@torch.no_grad()
def evaluate(n_classes=3):
    model.eval()
    pred = model(data.x, data.edge_index).argmax(1).cpu().numpy()
    true = data.y.cpu().numpy()
    mask = data.test_mask.cpu().numpy()
    p, t = pred[mask], true[mask]
    acc = (p == t).mean()
    ious = []
    for c in range(n_classes):
        inter = np.logical_and(p == c, t == c).sum()
        union = np.logical_or(p == c, t == c).sum()
        if union > 0:
            ious.append(inter / union)
    return acc, float(np.mean(ious)), ious

acc, miou, ious = evaluate()
print(f"Test accuracy : {acc:.3f}")
print(f"Test mIoU     : {miou:.3f}")
print(f"Per-class IoU : {[round(v,3) for v in ious]}")


## 7. Qualitative Results


In [ ]:
model.eval()
with torch.no_grad():
    pred = model(data.x, data.edge_index).argmax(1).cpu().numpy()

fig = plt.figure(figsize=(9, 4))
ax1 = fig.add_subplot(121, projection="3d")
plot_mesh_labels(V, Fc, y, "Ground truth", ax1)
ax2 = fig.add_subplot(122, projection="3d")
plot_mesh_labels(V, Fc, pred, "GNN prediction", ax2)
plt.tight_layout(); plt.show()


## 8. Isometry Robustness &mdash; Closing the Loop with Week 4

A central claim of the course is that **intrinsic features yield deformation-invariant predictions**. We test it: apply a random rigid rotation to the mesh, recompute the *same* intrinsic features, and confirm the segmentation is preserved &mdash; up to the sign/ordering ambiguity of numerically computed eigenfunctions &mdash; in stark contrast to the coordinate-based PointNet of Week 4.


In [ ]:
def random_rotation():
    A = np.random.randn(3,3); Q,R = np.linalg.qr(A)
    Q = Q @ np.diag(np.sign(np.diag(R)))
    if np.linalg.det(Q) < 0: Q[:,0] *= -1
    return Q

V_rot = V @ random_rotation().T
X_rot = intrinsic_features(V_rot, Fc)             # intrinsic -> should match
data_rot = mesh_to_graph(V_rot, Fc, X_rot, y).to(device)

with torch.no_grad():
    pred_rot = model(data_rot.x, data_rot.edge_index).argmax(1).cpu().numpy()

agreement = (pred_rot == pred).mean()
print(f"Prediction agreement before vs. after rotation: {agreement:.3f}")
print(
    "Intrinsic (HKS / Laplace-Beltrami) features are isometry-invariant in theory, "
    "so the segmentation is largely preserved under rigid motion -- unlike the\n"
    "coordinate-based PointNet of Week 4. Residual disagreement comes from the\n"
    "sign/ordering ambiguity of numerically computed eigenfunctions on (near-)\n"
    "degenerate eigenvalues, not from the rotation itself."
)


## 9. Conclusion

We have built a complete geometric deep learning system that segments 3D meshes by:

1. Treating the mesh as a **graph** (Week 2).
2. Computing **isometry-invariant intrinsic features** &mdash; HKS and Laplace&ndash;Beltrami encodings (Week 5).
3. Applying a **permutation-equivariant GNN** for per-vertex prediction (Week 3).
4. Evaluating with **mIoU** and confirming **deformation robustness** (Week 4), all within the **GDL blueprint** of equivariant layers and symmetry-respecting design (Week 1).

This is geometric deep learning in microcosm: *identify the domain and its symmetries, engineer features and layers that respect them, and let the right priors do the heavy lifting.*

### Exercises / Extensions

1. Swap `GCNConv` for `GATConv` and compare mIoU. Does attention help on this task?
2. Run a **feature ablation**: train on raw coordinates only, then add HKS, then add eigenfunctions. Plot mIoU vs. feature set.
3. Generate a *multi-mesh* dataset with varying shapes and train an inductive model; evaluate generalisation to unseen meshes.
4. Replace the synthetic loader with **ShapeNet-Part** and report mIoU against published baselines.
